In [6]:
import paho.mqtt.client as mqtt
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import numpy as np
import time
from datetime import datetime
import os
import re

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
BROKER = "127.0.0.1"
MQTT_TOPIC = "imu/data"
RECORD_SECONDS = 10
CSV_FILE = "mqtt_messages_log.csv"

POSTGRES = {
    "host": "localhost",
    "port": 5432, 
    "database": "iotdata",
    "user": "postgres",
    "password": "password"
}


os.makedirs(".", exist_ok=True)

mqtt_packets = []
collecting = True


# ─────────────────────────────────────────────────────────────
# MQTT CALLBACKS
# ─────────────────────────────────────────────────────────────
def on_connect(client, userdata, flags, rc):
    print("Connected:", rc)
    client.subscribe(MQTT_TOPIC)

def on_message(client, userdata, msg):
    global mqtt_packets, collecting
    if not collecting:
        return

    payload = msg.payload.decode("utf-8")

    # Extract timestamp + sample_id
    # Expected: [[timestamp, ax, ay, az, gx, gy, gz]]
    match = re.findall(r"\[([0-9\.\-]+)", payload)
    if not match:
        return

    timestamp = int(match[0])  
    mqtt_packets.append({"timestamp": timestamp})

    print(f"Packets: {len(mqtt_packets)}", end="\r")


# ─────────────────────────────────────────────────────────────
# STEP 1: COLLECT MQTT MESSAGES
# ─────────────────────────────────────────────────────────────
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

print(f"📡 Collecting MQTT messages for {RECORD_SECONDS}s...")
client.connect(BROKER, 1883, 60)
client.loop_start()

start = time.time()
while time.time() - start < RECORD_SECONDS:
    time.sleep(0.05)

collecting = False
client.loop_stop()
client.disconnect()

print(f"\nDone. Total packets collected: {len(mqtt_packets)}")

if len(mqtt_packets) == 0:
    raise ValueError("No MQTT data collected — cannot continue.")


# ─────────────────────────────────────────────────────────────
# STEP 2: SAVE CSV
# ─────────────────────────────────────────────────────────────
mqtt_df = pd.DataFrame(mqtt_packets)
mqtt_df["sample_id"] = mqtt_df.index

mqtt_df.to_csv(CSV_FILE, index=False)
print(f"💾 Saved to: {CSV_FILE}")

# ─────────────────────────────────────────────────────────────
# STEP 3: LOAD FROM POSTGRES
# ─────────────────────────────────────────────────────────────
conn = psycopg2.connect(**POSTGRES)
cursor = conn.cursor()

cursor.execute("SELECT sample_id, timestamp FROM imu_data ORDER BY sample_id ASC;")
rows = cursor.fetchall()
conn.close()

db_df = pd.DataFrame(rows, columns=["sample_id", "timestamp"])
print(f"📥 Rows in DB: {len(db_df)}")

# ─────────────────────────────────────────────────────────────
# STEP 4: COMPARE MQTT vs DB
# ─────────────────────────────────────────────────────────────
sent_ids = set(mqtt_df["sample_id"])
stored_ids = set(db_df["sample_id"])

lost_ids = sent_ids - stored_ids
loss_rate = (len(lost_ids) / len(sent_ids)) * 100

print("\n──── Comparison Summary ────")
print(f"Sent (MQTT):   {len(sent_ids)}")
print(f"Stored (DB):   {len(stored_ids)}")
print(f"Lost:          {len(lost_ids)}")
print(f"Loss rate:     {loss_rate:.2f}%")


# ─────────────────────────────────────────────────────────────
# STEP 5: PLOTS
# ─────────────────────────────────────────────────────────────

# Gaps inside MQTT timestamps (sampling jitter)
mqtt_df["dt"] = mqtt_df["timestamp"].diff()
mqtt_df.loc[mqtt_df["dt"] < 0, "dt"] += 2**32




/var/folders/m1/qxpcclg55hg_8gcmt4vpkrq80000gn/T/ipykernel_25718/3939931685.py:63: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


📡 Collecting MQTT messages for 10s...
Connected: 0
Packets: 251
Done. Total packets collected: 251
💾 Saved to: mqtt_messages_log.csv
📥 Rows in DB: 13895

──── Comparison Summary ────
Sent (MQTT):   251
Stored (DB):   13831
Lost:          0
Loss rate:     0.00%
